In [1]:
import pandas as pd
import os
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

Cargamos el archivo `corpus.csv` y visualizamos las primeras filas para entender su estructura.

In [2]:
ruta_input = "corpus.csv"

if os.path.exists(ruta_input):
    df = pd.read_csv(ruta_input, encoding='utf-8')
    print(f"Se cargaron {len(df)} películas.")
    display(df.head(3))
else:
    print(f"Error: No se encontró el archivo '{ruta_input}' en el directorio.")

Se cargaron 70 películas.


,titulo,idioma,sinopsis,reseña,género
0,Avatar,Español,Exploramos en Avatar la historia de una serie ...,"Tras verla, diría que es un poco lenta, pero c...","Acción, Aventura, Fantasía, Ciencia Ficción"
1,Pirates of the Caribbean: At World's End,Inglés,Acompañamos a los personajes de Pirates of the...,"Desde mi punto de vista, estamos ante impactan...","Aventura, Fantasía, Acción"
2,Spectre,Español,Acompañamos a los personajes de Spectre en un ...,Una experiencia cinematográfica fascinante y m...,"Acción, Aventura, Crimen"


Para cada película, unimos los siguientes campos en un único texto:
- `titulo` (Título de la película)
- `idioma` (Idioma de la película)
- `sinopsis` (Sinopsis de la película)
- `reseña` (Reseña de los usuarios)
- `género` (Géneros de la película)

Reemplazamos posibles valores nulos (`NaN`) con cadenas vacías para evitar errores de concatenación.
Además, aplicamos técnicas de normalización usando NLTK (conversión a minúsculas, tokenización, eliminación de signos de puntuación y palabras vacías, y reducción a la raíz o stemming).

In [3]:
# Configurar para español
stop_words = set(stopwords.words('spanish'))
stemmer = SnowballStemmer('spanish')

# Rellenar valores nulos con una cadena vacía
df = df.fillna("")

def elimina_no_alfanumerico(tokens):
    return [re.sub(r'[^\w]', '', token)
            for token in tokens
            if re.search(r'\w', token)]

def elimina_stopwords(tokens):
    return [token for token in tokens if token not in stop_words]

def aplica_stemmer(tokens):
    return [stemmer.stem(token) for token in tokens]

def procesar_texto(texto):
    texto = texto.lower()
    # Tokenizar
    tokens = word_tokenize(texto)
    tokens = elimina_no_alfanumerico(tokens)
    tokens = elimina_stopwords(tokens)
    tokens = aplica_stemmer(tokens)
    
    return " ".join(tokens)

# Crear una columna con el texto concatenado
df['documento_crudo'] = (
    df['titulo'] + 
    ". " + df['idioma'] + 
    " " + df['sinopsis'] + 
    " " + df['reseña'] + 
    " " + df['género']
)

# Crear la columna 'documento' procesada para el motor de búsqueda
df['documento'] = df['documento_crudo'].apply(procesar_texto)

In [4]:
print(f"Película: {df['titulo'].iloc[0]}\n")
print("Documento generado para búsqueda:")
print(df['documento'].iloc[0])

Película: Avatar

Documento generado para búsqueda:
avat español explor avat histori seri event misteri mantien suspens final ideal amant cin mensaj potent tras verl dir lent actuacion brillant recomend accion aventur fantas cienci ficcion


Guardamos el DataFrame resultante en un nuevo archivo CSV.

In [5]:
ruta_output = "corpus_procesado.csv"
df.to_csv(ruta_output, index=False, encoding='utf-8')
print(f" El corpus procesado se guardó en: {ruta_output}")

 El corpus procesado se guardó en: corpus_procesado.csv


### Creación del Índice Invertido con Whoosh
Utilizamos la biblioteca `Whoosh` para construir un índice invertido. Definiremos un esquema con el título y el contenido procesado, crearemos un directorio para el índice y añadiremos cada documento del corpus.

In [6]:
import os
from whoosh.index import create_in
from whoosh.fields import Schema, TEXT, ID

# Definir el esquema del índice
schema = Schema(
    id=ID(stored=True, unique=True),
    titulo=TEXT(stored=True),
    idioma=TEXT(stored=True),
    sinopsis=TEXT(stored=True),
    reseña=TEXT(stored=True),
    genero=TEXT(stored=True),
    contenido=TEXT(stored=True) # Campo combinado procesado para búsqueda global
)

# Crear el directorio para el índice si no existe
index_dir = "indexdir"
if not os.path.exists(index_dir):
    os.mkdir(index_dir)

# Crear el índice
ix = create_in(index_dir, schema)

# Abrir un escritor para añadir documentos
writer = ix.writer()

# Iterar sobre el dataframe para añadir los documentos al índice
for i, row in df.iterrows():
    writer.add_document(
        id=str(i),
        titulo=str(row['titulo']),
        idioma=str(row['idioma']),
        sinopsis=str(row['sinopsis']),
        reseña=str(row['reseña']),
        genero=str(row['género']),
        contenido=str(row['documento'])
    )

writer.commit()
print("Índice invertido creado con éxito en el directorio 'indexdir'.")

Índice invertido creado con éxito en el directorio 'indexdir'.


### Sistema de Recuperación Booleano
A continuación, implementamos el sistema de recuperación booleano utilizando Whoosh. Se define una función para procesar la consulta del usuario (aplicando la misma normalización que a los documentos, pero preservando los operadores lógicos) y otra función para ejecutar la búsqueda y mostrar los resultados.

In [7]:
import re
from whoosh import qparser
from whoosh.index import open_dir
import pandas as pd

def procesar_query_booleana(query_str):
    """
    Procesa la consulta manteniendo los operadores lógicos booleanos intactos,
    pero aplicando el stemmer y eliminación de stopwords a los términos de búsqueda.
    """
    # Separar paréntesis para tratarlos como tokens independientes
    query_str = query_str.replace('(', ' ( ').replace(')', ' ) ')
    tokens = query_str.split()
    
    procesados = []
    for t in tokens:
        if t in ['AND', 'OR', 'NOT', '(', ')']:
            procesados.append(t)
        else:
            t_proc = procesar_texto(t)
            if t_proc:
                procesados.append(t_proc)
            
    return " ".join(procesados)

def buscar_booleano(query_str, index_dir="indexdir"):
    ix = open_dir(index_dir)
    
    query_procesada = procesar_query_booleana(query_str)
    print(f"Consulta original: {query_str}")
    print(f"Consulta procesada: {query_procesada}\n")
    
    with ix.searcher() as searcher:
        # Usamos el QueryParser en el campo 'contenido'
        parser = qparser.QueryParser("contenido", ix.schema)
        
        try:
            query = parser.parse(query_procesada)
            results = searcher.search(query, limit=None) # limit=None para devolver todos los compatibles
            
            print(f"Se encontraron {len(results)} documentos para la consulta.\n")
            for i, result in enumerate(results):
                print(f"--- Resultado {i+1} ---")
                print(f"Título: {result['titulo']}")
                print(f"Género: {result['genero']}")
                print(f"Idioma: {result['idioma']}")
                print("-" * 20)
                
        except Exception as e:
            print(f"Error al procesar la consulta: {e}")

# Cargamos las consultas desde el CSV del proyecto
df_consultas = pd.read_csv("consultas.csv")

# Probamos el buscador booleano con la Consulta #1 de nuestro dataset oficial
consulta_prueba_bool = df_consultas.iloc[0]['consulta_booleana']
buscar_booleano(consulta_prueba_bool)


Consulta original: ciencia AND ficción AND NOT romance
Consulta procesada: cienci AND ficcion AND NOT romanc

Se encontraron 28 documentos para la consulta.

--- Resultado 1 ---
Título: The Avengers
Género: Ciencia Ficción, Acción, Aventura
Idioma: Español
--------------------
--- Resultado 2 ---
Título: Battleship
Género: Suspense, Acción, Aventura, Ciencia Ficción
Idioma: Inglés
--------------------
--- Resultado 3 ---
Título: 2012
Género: Acción, Aventura, Ciencia Ficción
Idioma: Inglés
--------------------
--- Resultado 4 ---
Título: X-Men: Apocalypse
Género: Ciencia Ficción
Idioma: Español
--------------------
--- Resultado 5 ---
Título: Avatar
Género: Acción, Aventura, Fantasía, Ciencia Ficción
Idioma: Español
--------------------
--- Resultado 6 ---
Título: John Carter
Género: Acción, Aventura, Ciencia Ficción
Idioma: Español
--------------------
--- Resultado 7 ---
Título: Men in Black 3
Género: Acción, Comedia, Ciencia Ficción
Idioma: Español
--------------------
--- Resultado

### Sistema de Recuperación por Texto Libre (Ranking)
A continuación, implementamos el sistema de recuperación por texto libre utilizando Whoosh. Se utiliza el modelo TF-IDF y la similitud del coseno (configurable en Whoosh mediante `scoring.TF_IDF()`) para establecer un ranking entre los documentos devueltos en función de su similitud con la consulta.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from whoosh.index import open_dir
import pandas as pd

def buscar_texto_libre(query_str, index_dir="indexdir"):
    ix = open_dir(index_dir)
    
    # Procesamos la consulta de texto libre igual que los documentos
    query_procesada = procesar_texto(query_str)
    print(f"Consulta original: {query_str}")
    print(f"Consulta procesada: {query_procesada}\n")
    
    with ix.searcher() as searcher:
        # 1. Obtener todos los documentos almacenados en el índice
        documentos = list(searcher.all_stored_fields())
        textos = [doc['contenido'] for doc in documentos]
        
        # 2. Calcular TF-IDF y Similitud del Coseno
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(textos)
        query_vector = vectorizer.transform([query_procesada])
        
        similitudes = cosine_similarity(query_vector, tfidf_matrix).flatten()
        
        # 3. Obtener los índices ordenados de mayor a menor similitud
        indices_ordenados = np.argsort(similitudes)[::-1]
        
        # Contar cuántos documentos tienen similitud > 0
        encontrados = np.sum(similitudes > 0.00)
        print(f"Se encontraron {encontrados} documentos para la consulta.\n")
        
        # Mostrar resultados en orden de ranking
        ranking = 1
        for idx in indices_ordenados:
            score = similitudes[idx]
            if score > 0.00:
                doc = documentos[idx]
                print(f"--- Ranking {ranking} (Similitud Coseno: {score:.4f}) ---")
                print(f"Título: {doc['titulo']}")
                print(f"Género: {doc['genero']}")
                print(f"Idioma: {doc['idioma']}")
                print("-" * 20)
                ranking += 1

# Probamos el buscador vectorial con la Consulta de texto libre #1 de nuestro dataset oficial
consulta_prueba_text = df_consultas.iloc[0]['consulta_texto_libre']
buscar_texto_libre(consulta_prueba_text)


Consulta original: ciencia ficción no romance
Consulta procesada: cienci ficcion romanc

Se encontraron 31 documentos para la consulta.

--- Ranking 1 (Similitud Coseno: 0.1927) ---
Título: Titanic
Género: Drama, Romance, Suspense
Idioma: Inglés
--------------------
--- Ranking 2 (Similitud Coseno: 0.1615) ---
Título: The Great Gatsby
Género: Drama, Romance
Idioma: Inglés
--------------------
--- Ranking 3 (Similitud Coseno: 0.1320) ---
Título: Prince of Persia: The Sands of Time
Género: Aventura, Fantasía, Acción, Romance
Idioma: Español
--------------------
--- Ranking 4 (Similitud Coseno: 0.1018) ---
Título: The Avengers
Género: Ciencia Ficción, Acción, Aventura
Idioma: Español
--------------------
--- Ranking 5 (Similitud Coseno: 0.1007) ---
Título: 2012
Género: Acción, Aventura, Ciencia Ficción
Idioma: Inglés
--------------------
--- Ranking 6 (Similitud Coseno: 0.0967) ---
Título: Battleship
Género: Suspense, Acción, Aventura, Ciencia Ficción
Idioma: Inglés
--------------------
-

### Evaluación de los Sistemas de Recuperación (Métricas)
En esta sección evaluamos el rendimiento de ambos motores de búsqueda (Booleano y Vectorial/Coseno) utilizando las 20 consultas definidas en `consultas.csv` y sus correspondientes juicios de relevancia en `relevancia.csv`.

Calculamos:
1. **Precisión** y **Sensibilidad (Recall)** para el sistema Booleano.
2. **Average Precision (AP)** para cada consulta y el **MAP (Mean Average Precision)** global para el sistema Vectorial.

In [9]:
import pandas as pd
import numpy as np
from whoosh.index import open_dir
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def obtener_resultados_booleanos(query_str, index_dir="indexdir"):
    ix = open_dir(index_dir)
    query_procesada = procesar_query_booleana(query_str)
    retrieved_ids = set()
    with ix.searcher() as searcher:
        parser = qparser.QueryParser("contenido", ix.schema)
        query = parser.parse(query_procesada)
        results = searcher.search(query, limit=None)
        for r in results:
            retrieved_ids.add(int(r['id']))
    return retrieved_ids

def obtener_resultados_coseno(query_str, documentos):
    query_procesada = procesar_texto(query_str)
    textos = [doc['contenido'] for doc in documentos]
    
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(textos)
    query_vector = vectorizer.transform([query_procesada])
    
    similitudes = cosine_similarity(query_vector, tfidf_matrix).flatten()
    indices_ordenados = np.argsort(similitudes)[::-1]
    
    ranked_ids = []
    for idx in indices_ordenados:
        score = similitudes[idx]
        if score > 0.00:
            doc = documentos[idx]
            ranked_ids.append(int(doc['id']))
    return ranked_ids

# Cargar consultas y relevancia
df_consultas = pd.read_csv("consultas.csv")
df_relevancia = pd.read_csv("relevancia.csv")

# Cargar todos los documentos del indice
ix = open_dir("indexdir")
with ix.searcher() as searcher:
    documentos = list(searcher.all_stored_fields())

resultados_eval = []

# Bucle para evaluar cada consulta
for index, row in df_consultas.iterrows():
    need_id = int(row['necesidad_id'])
    query_bool = row['consulta_booleana']
    query_text = row['consulta_texto_libre']
    
    # Relevantes reales de la consulta actual
    relevantes = set(df_relevancia[(df_relevancia['necesidad_id'] == need_id) & (df_relevancia['relevante'] == 1)]['documento_id'].tolist())
    
    # Evaluacion Booleana (Precision y Recall)
    recuperados_bool = obtener_resultados_booleanos(query_bool)
    if len(recuperados_bool) > 0:
        precision = len(recuperados_bool.intersection(relevantes)) / len(recuperados_bool)
    else:
        precision = 0.0
        
    if len(relevantes) > 0:
        recall = len(recuperados_bool.intersection(relevantes)) / len(relevantes)
    else:
        recall = 0.0
        
    # Evaluacion Coseno (AP)
    recuperados_cos = obtener_resultados_coseno(query_text, documentos)
    hits = 0
    sum_precisions = 0.0
    for k, doc_id in enumerate(recuperados_cos, 1):
        if doc_id in relevantes:
            hits += 1
            sum_precisions += hits / k
            
    if len(relevantes) > 0:
        ap = sum_precisions / len(relevantes)
    else:
        ap = 0.0
        
    resultados_eval.append({
        "Necesidad": need_id,
        "Precision Booleana": precision,
        "Recall Booleano": recall,
        "AP Coseno": ap
    })
    
df_eval = pd.DataFrame(resultados_eval)

# Medias globales de las metricas
mean_prec = df_eval['Precision Booleana'].mean()
mean_rec = df_eval['Recall Booleano'].mean()
map_coseno = df_eval['AP Coseno'].mean()

print("Precision Booleana Promedio:", round(mean_prec, 4))
print("Recall Booleano Promedio:", round(mean_rec, 4))
print("MAP Coseno:", round(map_coseno, 4))
print()

display(df_eval.round(4))


Precision Booleana Promedio: 0.998
Recall Booleano Promedio: 1.0
MAP Coseno: 0.7248



,Necesidad,Precision Booleana,Recall Booleano,AP Coseno
0,1,1.00,1.0,0.7649
1,2,1.00,1.0,0.4289
2,3,1.00,1.0,0.8752
3,4,1.00,1.0,0.2766
4,5,1.00,1.0,0.6679
5,6,0.96,1.0,0.6075
6,7,1.00,1.0,1.0000
7,8,1.00,1.0,1.0000
8,9,1.00,1.0,1.0000
9,10,1.00,1.0,0.5429
